In [35]:
# loading in necessary libraries and cleaned data from last file
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df_clean = pd.read_csv('data/cleaned_survey.csv')
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 4448 entries, 0 to 4447
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ResponseId              4448 non-null   int64  
 1   Age                     4448 non-null   str    
 2   EdLevel                 4448 non-null   str    
 3   Employment              4448 non-null   str    
 4   WorkExp                 4448 non-null   float64
 5   YearsCode               4448 non-null   float64
 6   DevType                 4448 non-null   str    
 7   OrgSize                 4448 non-null   str    
 8   ICorPM                  4448 non-null   str    
 9   RemoteWork              4448 non-null   str    
 10  Industry                4448 non-null   str    
 11  Country                 4448 non-null   str    
 12  LanguageHaveWorkedWith  4448 non-null   str    
 13  DatabaseHaveWorkedWith  3760 non-null   str    
 14  annual_salary_usd       4448 non-null   float64
 15

# ENCODING & MODELING
### (picks up where exploredata.ipynb left off, trying to avoid lengthiness)

## ENCODING 'Age', 'EdLevel', and 'OrgSize'

In [36]:
# idea is to ordinal encode, encoding over ordered integers (e.g. 1, 2, 3, 4, ...)
# diff than one-hot, which is binary
df_clean['Age'].value_counts()

Age
25-34 years old      1686
35-44 years old      1426
45-54 years old       604
18-24 years old       439
55-64 years old       246
65 years or older      42
Prefer not to say       5
Name: count, dtype: int64

In [37]:
# prefer not to say has only 5 respondants
# deciding to drop those rows instead of forcing those responses into a numerical category
df_clean = df_clean[df_clean['Age'] != 'Prefer not to say']
print(f'Rows Remaining: {len(df_clean)}')

Rows Remaining: 4443


In [38]:
# ordinal encoding for ages
age_map = {
    '18-24 years old': 1,
    '25-34 years old': 2,
    '35-44 years old': 3,
    '45-54 years old': 4,
    '55-64 years old': 5,
    '65 years or older': 6
}
df_clean['Age_encoded'] = df_clean['Age'].map(age_map)

In [39]:
# also intending to ordinal encode 'EdLevel', but looking at values first
# number of respondants who put 'Other' feels like more substantial than the respondants for 'prefer not to say' for Age
# primary/elementary school feels concerning though, maybe they didn't complete and were self-taught?
df_clean['EdLevel'].value_counts()

EdLevel
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          2024
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                                       1321
Some college/university study without earning a degree                                 516
Professional degree (JD, MD, Ph.D, Ed.D, etc.)                                         216
Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)     181
Associate degree (A.A., A.S., etc.)                                                    135
Other (please specify):                                                                 35
Primary/elementary school                                                               15
Name: count, dtype: int64

In [40]:
# think i'm going to drop respondants who said 'primary/elemantary school', lowest number of respondants as well
df_clean = df_clean[df_clean['EdLevel'] != 'Primary/elementary school']
print(f'Rows Remaining: {len(df_clean)}')

Rows Remaining: 4428


In [41]:
# ordinal encoding for education levels
edlevel_map = {
    'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 1,
    'Some college/university study without earning a degree': 2,
    'Associate degree (A.A., A.S., etc.)': 3,
    "Bachelor’s degree (B.A., B.S., B.Eng., etc.)": 4,
    "Master’s degree (M.A., M.S., M.Eng., MBA, etc.)": 5,
    'Professional degree (JD, MD, Ph.D, Ed.D, etc.)': 6,
    'Other (please specify):': 0
}
df_clean['EdLevel_encoded'] = df_clean['EdLevel'].map(edlevel_map)

In [42]:
# intending to ordinal encode 'OrgSize' too, inspecting first
# all seem relevant and substantial, keeping all
df_clean['OrgSize'].value_counts()

OrgSize
20 to 99 employees                                    1316
100 to 499 employees                                   808
Less than 20 employees                                 643
10,000 or more employees                               597
1,000 to 4,999 employees                               522
500 to 999 employees                                   267
5,000 to 9,999 employees                               176
Just me - I am a freelancer, sole proprietor, etc.      99
Name: count, dtype: int64

In [43]:
# ordinal encoding for orgization size
orgsize_map = {
    'Just me - I am a freelancer, sole proprietor, etc.': 1,
    'Less than 20 employees': 2,
    '20 to 99 employees': 3,
    '100 to 499 employees': 4,
    '500 to 999 employees': 5,
    '1,000 to 4,999 employees': 6,
    '5,000 to 9,999 employees': 7,
    '10,000 or more employees': 8
}
df_clean['OrgSize_encoded'] = df_clean['OrgSize'].map(orgsize_map)

In [44]:
# checking to see if everything mapped over alright, should be no null values
# YAY!!
print(df_clean[['Age_encoded', 'EdLevel_encoded', 'OrgSize_encoded']].isna().sum())

Age_encoded        0
EdLevel_encoded    0
OrgSize_encoded    0
dtype: int64


In [45]:
# for nominal/categorical columns, choosing to one-hot encode
# no hierarchy where one thing is "more" or "less" than another
# each will get its own category of whether it's "filled" or not
nominal_columns = ['DevType', 'Country', 'Industry', 'Employment', 'ICorPM', 'RemoteWork']

df_encoded = pd.get_dummies(df_clean, columns = nominal_columns, drop_first = True)

In [46]:
# need to multilabel encoded for languges and databases use, since more than one instance can belong to a cell
# semicolon separated values
language_dummies = df_clean['LanguageHaveWorkedWith'].str.get_dummies(sep=';').add_prefix('lang_')
database_dummies = df_clean['DatabaseHaveWorkedWith'].str.get_dummies(sep=';').add_prefix('db_')

df_encoded = pd.concat([df_encoded.drop(columns = ['LanguageHaveWorkedWith', 'DatabaseHaveWorkedWith']), 
                        language_dummies, database_dummies], axis = 1)

In [47]:
# log encoding the target value (annual_salary_usd) to reduce right-skew we saw from earlier
df_encoded['log_salary'] = np.log1p(df_encoded['annual_salary_usd'])

In [48]:
df_encoded = df_encoded.drop(columns = ['Age', 'EdLevel', 'OrgSize'])

## ENCODING DONE!

## Train/Test & Modelling

In [54]:
# Check which columns contain missing values
missing = X.isna().sum()
print(missing[missing > 0])

max_reasonable_age    42
dtype: int64


In [55]:
# training on one portion of data, testing on unseen data to evaluate model's effectiveness at generalizing

from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns = ['annual_salary_usd', 'log_salary', 'ResponseId', 'max_reasonable_age'])
y = df_encoded['log_salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

Train: 3542, Test: 886


In [62]:
# fitting a baseline model, going with linear regression
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

## Evaluating R^2 and some stats

In [63]:
# evaluating the model to see if any changes should be made
# changes will have to be made, prob a different model might work
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f'R2: {r2:.3f}')
print(f'MAE (log scale): {mae:.3f}')
print(f'RMSE (log scale): {rmse:.3f}')

R2: 0.448
MAE (log scale): 0.518
RMSE (log scale): 0.865


In [ ]:
# want to show error in real dollars, more easily interpretable
